In [2]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 1 - Environment and Google Drive Preflight
# ============================================================

import os
import sys
import platform
import subprocess
from pathlib import Path

import torch
from google.colab import drive


print("=" * 72)
print("MAIA LITE - SFT A100")
print("ENVIRONMENT AND GOOGLE DRIVE PREFLIGHT")
print("=" * 72)
print()


# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

DRIVE_MOUNT = "/content/drive"

print("Mounting Google Drive...")
drive.mount(DRIVE_MOUNT)

print()


# ------------------------------------------------------------
# 2. Project paths
# ------------------------------------------------------------

COLAB_ROOT = Path(
    "/content/drive/MyDrive/Colab_LLMs"
)

SFT_PROJECT = COLAB_ROOT / "maia-sft-validation"

MAIA_PROJECT = COLAB_ROOT / "Maia_Lite_v3"

MAIA_FINAL_CHECKPOINT = (
    MAIA_PROJECT
    / "checkpoints"
    / "checkpoint-216567"
)

STAGE4_DIR = SFT_PROJECT / "data" / "stage4"

BENCHMARK_DIR = STAGE4_DIR / "benchmark"

OUTPUT_DIR = SFT_PROJECT / "outputs"


print("Paths")
print("-" * 72)
print(f"Colab root:            {COLAB_ROOT}")
print(f"SFT project:           {SFT_PROJECT}")
print(f"Maia project:          {MAIA_PROJECT}")
print(f"Maia final checkpoint: {MAIA_FINAL_CHECKPOINT}")
print(f"Stage 4 data:          {STAGE4_DIR}")
print(f"Output directory:      {OUTPUT_DIR}")
print()


# ------------------------------------------------------------
# 3. Environment information
# ------------------------------------------------------------

print("Environment")
print("-" * 72)

print(f"Python:   {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.version.cuda}")
print()


# ------------------------------------------------------------
# 4. CUDA / GPU verification
# ------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. "
        "Stop here and select an A100 runtime."
    )

device = torch.device("cuda")

gpu_name = torch.cuda.get_device_name(0)

properties = torch.cuda.get_device_properties(0)

gpu_memory_gib = properties.total_memory / (1024 ** 3)

print("GPU")
print("-" * 72)
print(f"Device:       {gpu_name}")
print(f"Memory:       {gpu_memory_gib:.2f} GiB")
print(f"BF16 support: {torch.cuda.is_bf16_supported()}")
print()


# ------------------------------------------------------------
# 5. Require A100
# ------------------------------------------------------------

if "A100" not in gpu_name.upper():
    raise RuntimeError(
        f"Expected NVIDIA A100, but detected: {gpu_name}"
    )

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "The selected GPU does not report BF16 support."
    )


# ------------------------------------------------------------
# 6. Verify required directories
# ------------------------------------------------------------

print("Directory verification")
print("-" * 72)

required_directories = [
    COLAB_ROOT,
    SFT_PROJECT,
    STAGE4_DIR,
    BENCHMARK_DIR,
    OUTPUT_DIR,
]

directory_failure = False

for path in required_directories:
    if path.is_dir():
        print(f"OK       {path}")
    else:
        print(f"MISSING  {path}")
        directory_failure = True

print()

if directory_failure:
    raise RuntimeError(
        "One or more required directories are missing."
    )


# ------------------------------------------------------------
# 7. Verify Stage 4 critical files
# ------------------------------------------------------------

required_files = [
    STAGE4_DIR / "train.jsonl",
    STAGE4_DIR / "validation.jsonl",
    STAGE4_DIR / "test.jsonl",
    BENCHMARK_DIR / "stage4-v4-benchmark.jsonl",
    SFT_PROJECT / "src" / "train_stage4.py",
    SFT_PROJECT / "src" / "train_stage4_lora.py",
    SFT_PROJECT / "requirements-stage4.txt",
]

print("Stage 4 file verification")
print("-" * 72)

file_failure = False

for path in required_files:
    if path.is_file():
        size_mib = path.stat().st_size / (1024 ** 2)
        print(
            f"OK       {path.relative_to(SFT_PROJECT)} "
            f"({size_mib:.2f} MiB)"
        )
    else:
        print(
            f"MISSING  {path.relative_to(SFT_PROJECT)}"
        )
        file_failure = True

print()

if file_failure:
    raise RuntimeError(
        "One or more required Stage 4 files are missing."
    )


# ------------------------------------------------------------
# 8. NVIDIA runtime information
# ------------------------------------------------------------

print("NVIDIA runtime")
print("-" * 72)

subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    check=True,
)

print()


# ------------------------------------------------------------
# 9. Final status
# ------------------------------------------------------------

print("=" * 72)
print("PREFLIGHT CELL 1: PASS")
print("=" * 72)
print()
print("No training was started.")
print("No project files were modified.")

MAIA LITE - SFT A100
ENVIRONMENT AND GOOGLE DRIVE PREFLIGHT

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Paths
------------------------------------------------------------------------
Colab root:            /content/drive/MyDrive/Colab_LLMs
SFT project:           /content/drive/MyDrive/Colab_LLMs/maia-sft-validation
Maia project:          /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3
Maia final checkpoint: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/checkpoints/checkpoint-216567
Stage 4 data:          /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/data/stage4
Output directory:      /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs

Environment
------------------------------------------------------------------------
Python:   3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch:  2.11.0+cu130
CUDA:     13.0

GPU
---------------------------------

In [4]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 2 - Frozen Stage 4 SHA-256 Verification
# ============================================================

import hashlib
from pathlib import Path


print("=" * 72)
print("MAIA LITE - SFT A100")
print("FROZEN STAGE 4 SHA-256 VERIFICATION")
print("=" * 72)
print()


SFT_PROJECT = Path(
    "/content/drive/MyDrive/Colab_LLMs/maia-sft-validation"
)

STAGE4_DIR = SFT_PROJECT / "data" / "stage4"


# ------------------------------------------------------------
# Official frozen Stage 4 hashes
# ------------------------------------------------------------

EXPECTED_HASHES = {
    "train.jsonl":
        "ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9",

    "validation.jsonl":
        "5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a",

    "test.jsonl":
        "03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834",

    "benchmark/stage4-v4-benchmark.jsonl":
        "9ba60ac68ad285fdb0e16eb2af876722d8745e8131203b4a090b51b6a5ad73c3",
}


# ------------------------------------------------------------
# SHA-256 helper
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

failed = False

for relative_path, expected_hash in EXPECTED_HASHES.items():

    path = STAGE4_DIR / relative_path

    print("-" * 72)
    print(relative_path)

    if not path.is_file():
        print("STATUS:   MISSING")
        failed = True
        continue

    actual_hash = sha256_file(path)

    print(f"Expected: {expected_hash}")
    print(f"Actual:   {actual_hash}")

    if actual_hash == expected_hash:
        print("STATUS:   PASS")
    else:
        print("STATUS:   FAIL")
        failed = True


print()
print("=" * 72)

if failed:
    print("STAGE 4 INTEGRITY VERIFICATION: FAILED")
    print("=" * 72)

    raise RuntimeError(
        "Frozen Stage 4 integrity verification failed. "
        "Do not start training."
    )

else:
    print("STAGE 4 INTEGRITY VERIFICATION: PASS")
    print("=" * 72)
    print()
    print("Frozen Stage 4 artifacts are byte-identical.")
    print("Training may proceed only after the remaining preflight checks.")

MAIA LITE - SFT A100
FROZEN STAGE 4 SHA-256 VERIFICATION

------------------------------------------------------------------------
train.jsonl
Expected: ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9
Actual:   ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9
STATUS:   PASS
------------------------------------------------------------------------
validation.jsonl
Expected: 5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a
Actual:   5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a
STATUS:   PASS
------------------------------------------------------------------------
test.jsonl
Expected: 03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834
Actual:   03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834
STATUS:   PASS
------------------------------------------------------------------------
benchmark/stage4-v4-benchmark.jsonl
Expected: 9ba60ac68ad285fdb0e16eb2af876722d8745e8131203b4a090b51b6a5ad73c3
Actu

In [5]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 3 - Python Dependency Audit
#
# READ-ONLY:
#   - Does not install packages
#   - Does not modify project files
#   - Does not start training
# ============================================================

import sys
import importlib
import importlib.metadata

print("=" * 72)
print("MAIA LITE - SFT A100")
print("PYTHON DEPENDENCY AUDIT")
print("=" * 72)
print()

print(f"Python: {sys.version}")
print()

packages = [
    "torch",
    "transformers",
    "datasets",
    "peft",
    "accelerate",
    "safetensors",
    "tokenizers",
    "tqdm",
    "numpy",
]

print("Installed packages")
print("-" * 72)

missing = []

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package:16s} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package:16s} MISSING")
        missing.append(package)

print()


# ------------------------------------------------------------
# Import verification
# ------------------------------------------------------------

print("Import verification")
print("-" * 72)

import_failed = []

for package in packages:
    try:
        importlib.import_module(package)
        print(f"{package:16s} PASS")
    except Exception as exc:
        print(
            f"{package:16s} FAIL "
            f"({type(exc).__name__}: {exc})"
        )
        import_failed.append(package)

print()


# ------------------------------------------------------------
# PyTorch / CUDA verification
# ------------------------------------------------------------

import torch

print("PyTorch / CUDA")
print("-" * 72)

print(f"PyTorch version:       {torch.__version__}")
print(f"CUDA runtime:          {torch.version.cuda}")
print(f"CUDA available:        {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:                   {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported:        {torch.cuda.is_bf16_supported()}")
    print(
        f"GPU memory:            "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB"
    )

print()


# ------------------------------------------------------------
# Requirements file
# ------------------------------------------------------------

requirements_file = (
    SFT_PROJECT / "requirements-stage4.txt"
)

print("Project requirements")
print("-" * 72)

if requirements_file.is_file():
    print(requirements_file)
    print()

    contents = requirements_file.read_text(
        encoding="utf-8"
    )

    print(contents)
else:
    print("requirements-stage4.txt: MISSING")

print()


# ------------------------------------------------------------
# Final audit status
# ------------------------------------------------------------

print("=" * 72)

if missing or import_failed:
    print("DEPENDENCY AUDIT: ATTENTION REQUIRED")

    if missing:
        print(
            "Missing packages:",
            ", ".join(missing)
        )

    if import_failed:
        print(
            "Import failures:",
            ", ".join(import_failed)
        )
else:
    print("DEPENDENCY AUDIT: PASS")

print("=" * 72)
print()
print("No packages were installed.")
print("No project files were modified.")
print("No training was started.")

MAIA LITE - SFT A100
PYTHON DEPENDENCY AUDIT

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Installed packages
------------------------------------------------------------------------
torch            2.11.0+cu130
transformers     5.17.0
datasets         4.8.5
peft             0.21.0
accelerate       1.15.0
safetensors      0.8.0
tokenizers       0.23.2
tqdm             4.67.3
numpy            2.1.3

Import verification
------------------------------------------------------------------------
torch            PASS
transformers     PASS
datasets         PASS
peft             PASS
accelerate       PASS
safetensors      PASS
tokenizers       PASS
tqdm             PASS
numpy            PASS

PyTorch / CUDA
------------------------------------------------------------------------
PyTorch version:       2.11.0+cu130
CUDA runtime:          13.0
CUDA available:        True
GPU:                   NVIDIA A100-SXM4-80GB
BF16 supported:        True
GPU memory:            79.25 GiB

Pro

In [6]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 4 - Stage 4 Training Code Audit
#
# READ-ONLY:
#   - Does not modify scripts
#   - Does not start training
# ============================================================

from pathlib import Path
import hashlib
import re


print("=" * 72)
print("MAIA LITE - SFT A100")
print("STAGE 4 TRAINING CODE AUDIT")
print("=" * 72)
print()


scripts = [
    SFT_PROJECT / "src" / "train_stage4.py",
    SFT_PROJECT / "src" / "train_stage4_lora.py",
    SFT_PROJECT / "src" / "train_stage4_lora_mps_safe.py",
]


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


patterns = {
    "MPS references": r"\bmps\b|torch\.mps|empty_cache",
    "CUDA references": r"\bcuda\b|torch\.cuda",
    "BF16 references": r"\bbf16\b|bfloat16",
    "FP16 references": r"\bfp16\b|float16",
    "Gradient checkpointing": r"gradient_checkpoint",
    "use_cache": r"use_cache",
    "LoRA": r"\bLoraConfig\b|\bLoRA\b|\blora\b",
    "AdamW": r"\bAdamW\b",
    "batch size": r"batch_size",
    "gradient accumulation": r"gradient_accum",
}


for path in scripts:

    print("-" * 72)
    print(path.name)
    print("-" * 72)

    if not path.is_file():
        print("MISSING")
        print()
        continue

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    lines = text.splitlines()

    print(f"Size:    {path.stat().st_size:,} bytes")
    print(f"Lines:   {len(lines):,}")
    print(f"SHA-256: {sha256_file(path)}")
    print()

    print("Feature scan:")

    for label, pattern in patterns.items():
        matches = [
            index + 1
            for index, line in enumerate(lines)
            if re.search(
                pattern,
                line,
                flags=re.IGNORECASE,
            )
        ]

        if matches:
            preview = ", ".join(
                str(x) for x in matches[:12]
            )

            if len(matches) > 12:
                preview += ", ..."

            print(
                f"  {label:24s} "
                f"YES  lines: {preview}"
            )
        else:
            print(
                f"  {label:24s} NO"
            )

    print()


print("=" * 72)
print("CODE AUDIT COMPLETE")
print("=" * 72)
print()
print("No training script was executed.")
print("No project file was modified.")

MAIA LITE - SFT A100
STAGE 4 TRAINING CODE AUDIT

------------------------------------------------------------------------
train_stage4.py
------------------------------------------------------------------------
Size:    32,669 bytes
Lines:   1,533
SHA-256: 4a22a08fc9d66fc46924016fa126f728fc79f0a0057c2c6660665e931a405f6a

Feature scan:
  MPS references           YES  lines: 46, 190, 192, 194, 199, 200, 231, 232, 234, 239, 240
  CUDA references          YES  lines: 185, 186, 187, 223, 224, 226, 227
  BF16 references          NO
  FP16 references          NO
  Gradient checkpointing   NO
  use_cache                NO
  LoRA                     NO
  AdamW                    YES  lines: 934
  batch size               YES  lines: 272, 285
  gradient accumulation    YES  lines: 508, 510, 750

------------------------------------------------------------------------
train_stage4_lora.py
------------------------------------------------------------------------
Size:    37,343 bytes
Lines:   1,68

In [7]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 5 - Frozen Training Configuration Inspection
#
# READ-ONLY:
#   - Does not import training scripts
#   - Does not modify files
#   - Does not start training
# ============================================================

from pathlib import Path
import re


print("=" * 72)
print("MAIA LITE - SFT A100")
print("FROZEN TRAINING CONFIGURATION INSPECTION")
print("=" * 72)
print()


scripts = [
    SFT_PROJECT / "src" / "common_stage4.py",
    SFT_PROJECT / "src" / "train_stage4.py",
    SFT_PROJECT / "src" / "train_stage4_lora.py",
]


interesting_patterns = [
    r"CFG\s*=",
    r"model_id",
    r"max_length",
    r"batch_size",
    r"gradient_accumulation",
    r"learning_rate",
    r"weight_decay",
    r"epochs",
    r"warmup",
    r"max_grad_norm",
    r"SEED",
    r"def device",
    r"DataLoader",
    r"num_workers",
    r"pin_memory",
    r"gradient_checkpoint",
    r"use_cache",
    r"LORA_",
]


for path in scripts:

    print()
    print("=" * 72)
    print(path.name)
    print("=" * 72)

    if not path.is_file():
        print("MISSING")
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    matched_lines = set()

    for index, line in enumerate(lines):

        if any(
            re.search(pattern, line)
            for pattern in interesting_patterns
        ):
            # Include surrounding lines so multiline
            # dictionaries/configurations remain readable.
            start = max(0, index - 3)
            end = min(len(lines), index + 8)

            matched_lines.update(
                range(start, end)
            )

    if not matched_lines:
        print("No relevant configuration lines found.")
        continue

    previous = None

    for index in sorted(matched_lines):

        if (
            previous is not None
            and index > previous + 1
        ):
            print("    ...")

        print(
            f"{index + 1:5d}: "
            f"{lines[index]}"
        )

        previous = index


print()
print("=" * 72)
print("CONFIGURATION INSPECTION COMPLETE")
print("=" * 72)
print()
print("No training script was imported.")
print("No project file was modified.")
print("No training was started.")

MAIA LITE - SFT A100
FROZEN TRAINING CONFIGURATION INSPECTION


common_stage4.py
    1: import hashlib,json,random,re
    2: from pathlib import Path
    3: import torch
    4: ROOT=Path(__file__).resolve().parents[1]; CFG=json.loads((ROOT/"configs/stage4.json").read_text()); DATA=ROOT/"data/stage4"; OUT=ROOT/"outputs"; SEED=CFG["seed"]
    5: SYSTEM="You are a helpful, careful multilingual assistant. Answer in the user's language unless asked otherwise."
    6: def seed_all(): random.seed(SEED); torch.manual_seed(SEED)
    7: def device():
    8:     if torch.cuda.is_available(): return torch.device("cuda")
    9:     if torch.backends.mps.is_available(): return torch.device("mps")
   10:     return torch.device("cpu")
   11: def sha256(p):
   12:     h=hashlib.sha256()
   13:     with open(p,"rb") as f:
   14:         for b in iter(lambda:f.read(1048576),b""): h.update(b)

train_stage4.py
    8: from pathlib import Path
    9: 
   10: import torch
   11: from torch.utils.data import 

In [8]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 6 - Exact Frozen Stage 4 Configuration
# ============================================================

import json

config_path = (
    SFT_PROJECT
    / "configs"
    / "stage4.json"
)

print("=" * 72)
print("FROZEN STAGE 4 CONFIGURATION")
print("=" * 72)
print()

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

print()
print("=" * 72)
print("CONFIGURATION READ COMPLETE")
print("=" * 72)
print()
print("No project file was modified.")
print("No training was started.")

FROZEN STAGE 4 CONFIGURATION

{
  "stage": "stage4-full-assistant-control-v1",
  "model_id": "openai-community/gpt2-medium",
  "seed": 42,
  "max_length": 1024,
  "micro_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "epochs": 3,
  "learning_rate": 2e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.05,
  "max_grad_norm": 1.0,
  "objective": "response-only causal language modeling",
  "selection": "best validation loss",
  "languages": [
    "en",
    "pt",
    "es"
  ],
  "target_examples": 18000,
  "benchmark_examples": 300
}

CONFIGURATION READ COMPLETE

No project file was modified.
No training was started.


In [9]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 7 - Create A100 LoRA Training Variant
#
# Creates:
#   src/train_stage4_lora_a100.py
#
# The original train_stage4_lora.py is NEVER modified.
# ============================================================

from pathlib import Path
import hashlib


print("=" * 72)
print("MAIA LITE - SFT A100")
print("CREATE GPT-2 MEDIUM LoRA A100 VARIANT")
print("=" * 72)
print()


SOURCE = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora.py"
)

TARGET = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora_a100.py"
)


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Verify source
# ------------------------------------------------------------

EXPECTED_SOURCE_SHA256 = (
    "f38ce575754c4da54bbdad4e39886960"
    "c05bc207e2a5c5aae8a74f3bf849a6c4"
)

if not SOURCE.is_file():
    raise RuntimeError(
        f"Source script not found: {SOURCE}"
    )

source_hash = sha256_file(SOURCE)

print("Source:")
print(f"  {SOURCE}")
print()
print("Expected SHA-256:")
print(f"  {EXPECTED_SOURCE_SHA256}")
print()
print("Actual SHA-256:")
print(f"  {source_hash}")
print()

if source_hash != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(
        "Source train_stage4_lora.py does not match "
        "the audited script. A100 variant was NOT created."
    )

print("Source integrity: PASS")
print()


# ------------------------------------------------------------
# Read exact audited source
# ------------------------------------------------------------

text = SOURCE.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# Controlled transformation 1:
# Dedicated A100 output directories.
#
# This prevents collision with M4 experiment artifacts.
# ------------------------------------------------------------

replacements = {
    'CHECKPOINT_ROOT = OUT / "stage4-lora-checkpoints"':
        'CHECKPOINT_ROOT = OUT / "a100-gpt2-medium-lora-checkpoints"',

    'BEST_DIR = OUT / "gpt2-medium-stage4-lora-best"':
        'BEST_DIR = OUT / "a100-gpt2-medium-lora-best"',

    'FINAL_DIR = OUT / "gpt2-medium-stage4-lora-final"':
        'FINAL_DIR = OUT / "a100-gpt2-medium-lora-final"',

    'METRICS_FILE = OUT / "stage4-lora-training-metrics.json"':
        'METRICS_FILE = OUT / "a100-gpt2-medium-lora-training-metrics.json"',
}


for old, new in replacements.items():

    count = text.count(old)

    if count != 1:
        raise RuntimeError(
            f"Expected exactly one occurrence of:\n{old}\n"
            f"Found: {count}\n"
            "A100 variant was NOT created."
        )

    text = text.replace(
        old,
        new,
        1,
    )


# ------------------------------------------------------------
# Controlled transformation 2:
# Require CUDA explicitly.
#
# We deliberately do NOT allow silent fallback to MPS/CPU.
# ------------------------------------------------------------

needle = """    dev = device()

    print()
"""

replacement = """    dev = device()

    if dev.type != "cuda":
        raise RuntimeError(
            f"A100 experiment requires CUDA, got: {dev}"
        )

    gpu_name = torch.cuda.get_device_name(0)

    if "A100" not in gpu_name.upper():
        raise RuntimeError(
            f"A100 experiment requires NVIDIA A100, got: {gpu_name}"
        )

    if not torch.cuda.is_bf16_supported():
        raise RuntimeError(
            "A100 experiment requires BF16 support."
        )

    print()
    print("CUDA device:", gpu_name)
    print(
        "CUDA memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB",
    )
    print("BF16 support:", torch.cuda.is_bf16_supported())

    print()
"""

count = text.count(needle)

if count != 1:
    raise RuntimeError(
        "Could not locate the unique device initialization "
        "block in the audited source."
    )

text = text.replace(
    needle,
    replacement,
    1,
)


# ------------------------------------------------------------
# Controlled transformation 3:
# Experiment banner only.
# ------------------------------------------------------------

text = text.replace(
    "GPT-2 MEDIUM RESPONSE-ONLY LoRA SFT",
    "GPT-2 MEDIUM RESPONSE-ONLY LoRA SFT - A100 CONTROL",
    1,
)


# ------------------------------------------------------------
# Write atomically
# ------------------------------------------------------------

temporary = TARGET.with_suffix(
    TARGET.suffix + ".tmp"
)

temporary.write_text(
    text,
    encoding="utf-8",
)

temporary.replace(TARGET)


# ------------------------------------------------------------
# Verify result
# ------------------------------------------------------------

target_hash = sha256_file(TARGET)

print("A100 variant created:")
print(f"  {TARGET}")
print()

print("SHA-256:")
print(f"  {target_hash}")
print()

print("Output isolation:")
print("  checkpoints : outputs/a100-gpt2-medium-lora-checkpoints")
print("  best        : outputs/a100-gpt2-medium-lora-best")
print("  final       : outputs/a100-gpt2-medium-lora-final")
print(
    "  metrics     : "
    "outputs/a100-gpt2-medium-lora-training-metrics.json"
)

print()
print("Frozen experimental parameters:")
print("  model                    : openai-community/gpt2-medium")
print("  seed                     : 42")
print("  max_length               : 1024")
print("  micro_batch_size         : 1")
print("  gradient accumulation    : 8")
print("  effective batch          : 8")
print("  epochs                   : 3")
print("  learning rate            : 2e-5")
print("  weight decay             : 0.01")
print("  warmup ratio             : 0.05")
print("  max grad norm            : 1.0")
print("  LoRA rank                : 16")
print("  LoRA alpha               : 32")
print("  LoRA dropout             : 0.05")
print("  LoRA targets             : c_attn, c_proj, c_fc")
print("  gradient checkpointing   : preserved")
print("  use_cache=False          : preserved")

print()
print("=" * 72)
print("A100 VARIANT CREATION: PASS")
print("=" * 72)
print()
print("The original M4 training scripts were not modified.")
print("Training was NOT started.")

MAIA LITE - SFT A100
CREATE GPT-2 MEDIUM LoRA A100 VARIANT

Source:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/src/train_stage4_lora.py

Expected SHA-256:
  f38ce575754c4da54bbdad4e39886960c05bc207e2a5c5aae8a74f3bf849a6c4

Actual SHA-256:
  f38ce575754c4da54bbdad4e39886960c05bc207e2a5c5aae8a74f3bf849a6c4

Source integrity: PASS

A100 variant created:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/src/train_stage4_lora_a100.py

SHA-256:
  80edb17ae5c03c93cddc03db8bea3ad962908d9921d65d7b95c5f4e0398d8a4d

Output isolation:
  checkpoints : outputs/a100-gpt2-medium-lora-checkpoints
  best        : outputs/a100-gpt2-medium-lora-best
  final       : outputs/a100-gpt2-medium-lora-final
  metrics     : outputs/a100-gpt2-medium-lora-training-metrics.json

Frozen experimental parameters:
  model                    : openai-community/gpt2-medium
  seed                     : 42
  max_length               : 1024
  micro_batch_size         : 1
  gradient accumulation    : 8
  ef

In [10]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 8 - Validate A100 LoRA Variant
#
# READ-ONLY:
#   - Python syntax check
#   - Unified diff against audited source
#   - Safety checks
#   - Does NOT import or execute training code
# ============================================================

from pathlib import Path
import difflib
import hashlib
import py_compile


print("=" * 72)
print("MAIA LITE - SFT A100")
print("VALIDATE GPT-2 MEDIUM LoRA A100 VARIANT")
print("=" * 72)
print()


SOURCE = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora.py"
)

TARGET = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora_a100.py"
)


EXPECTED_SOURCE_SHA256 = (
    "f38ce575754c4da54bbdad4e39886960"
    "c05bc207e2a5c5aae8a74f3bf849a6c4"
)

EXPECTED_TARGET_SHA256 = (
    "80edb17ae5c03c93cddc03db8bea3ad9"
    "62908d9921d65d7b95c5f4e0398d8a4d"
)


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# 1. Integrity
# ------------------------------------------------------------

source_hash = sha256_file(SOURCE)
target_hash = sha256_file(TARGET)

print("Integrity")
print("-" * 72)

print(f"Source: {source_hash}")
print(f"Target: {target_hash}")

if source_hash != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(
        "Original audited source changed."
    )

if target_hash != EXPECTED_TARGET_SHA256:
    raise RuntimeError(
        "Generated A100 variant changed."
    )

print("STATUS: PASS")
print()


# ------------------------------------------------------------
# 2. Python syntax compilation
# ------------------------------------------------------------

print("Python syntax")
print("-" * 72)

try:
    py_compile.compile(
        str(TARGET),
        doraise=True,
    )
except py_compile.PyCompileError as exc:
    raise RuntimeError(
        f"A100 variant does not compile:\n{exc}"
    )

print("STATUS: PASS")
print()


# ------------------------------------------------------------
# 3. Read files
# ------------------------------------------------------------

source_lines = SOURCE.read_text(
    encoding="utf-8"
).splitlines(keepends=True)

target_lines = TARGET.read_text(
    encoding="utf-8"
).splitlines(keepends=True)

target_text = "".join(target_lines)


# ------------------------------------------------------------
# 4. Safety assertions
# ------------------------------------------------------------

print("Safety assertions")
print("-" * 72)

required_strings = [
    'a100-gpt2-medium-lora-checkpoints',
    'a100-gpt2-medium-lora-best',
    'a100-gpt2-medium-lora-final',
    'a100-gpt2-medium-lora-training-metrics.json',
    'A100 experiment requires CUDA',
    'A100 experiment requires NVIDIA A100',
    'A100 experiment requires BF16 support',
]

for value in required_strings:
    if value not in target_text:
        raise RuntimeError(
            f"Required A100 marker missing: {value}"
        )

    print(f"PASS  {value}")


for old_value in [
    'OUT / "stage4-lora-checkpoints"',
    'OUT / "gpt2-medium-stage4-lora-best"',
    'OUT / "gpt2-medium-stage4-lora-final"',
    'OUT / "stage4-lora-training-metrics.json"',
]:
    if old_value in target_text:
        raise RuntimeError(
            f"Old M4 output path remains active: {old_value}"
        )

print()
print("M4 output isolation: PASS")
print()


# ------------------------------------------------------------
# 5. Show complete unified diff
# ------------------------------------------------------------

print("=" * 72)
print("UNIFIED DIFF")
print("=" * 72)
print()

diff = list(
    difflib.unified_diff(
        source_lines,
        target_lines,
        fromfile="train_stage4_lora.py",
        tofile="train_stage4_lora_a100.py",
    )
)

if not diff:
    raise RuntimeError(
        "No differences found between source and A100 variant."
    )

print("".join(diff))


# ------------------------------------------------------------
# 6. Diff statistics
# ------------------------------------------------------------

added = sum(
    1
    for line in diff
    if line.startswith("+")
    and not line.startswith("+++")
)

removed = sum(
    1
    for line in diff
    if line.startswith("-")
    and not line.startswith("---")
)

print()
print("=" * 72)
print("DIFF STATISTICS")
print("=" * 72)
print(f"Added lines:   {added}")
print(f"Removed lines: {removed}")
print()


# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print("=" * 72)
print("A100 VARIANT STRUCTURAL VALIDATION: PASS")
print("=" * 72)
print()
print("Original M4 script integrity: PASS")
print("A100 script integrity:        PASS")
print("Python syntax:                PASS")
print("Output isolation:             PASS")
print()
print("Training was NOT started.")

MAIA LITE - SFT A100
VALIDATE GPT-2 MEDIUM LoRA A100 VARIANT

Integrity
------------------------------------------------------------------------
Source: f38ce575754c4da54bbdad4e39886960c05bc207e2a5c5aae8a74f3bf849a6c4
Target: 80edb17ae5c03c93cddc03db8bea3ad962908d9921d65d7b95c5f4e0398d8a4d
STATUS: PASS

Python syntax
------------------------------------------------------------------------
STATUS: PASS

Safety assertions
------------------------------------------------------------------------
PASS  a100-gpt2-medium-lora-checkpoints
PASS  a100-gpt2-medium-lora-best
PASS  a100-gpt2-medium-lora-final
PASS  a100-gpt2-medium-lora-training-metrics.json
PASS  A100 experiment requires CUDA
PASS  A100 experiment requires NVIDIA A100
PASS  A100 experiment requires BF16 support

M4 output isolation: PASS

UNIFIED DIFF

--- train_stage4_lora.py
+++ train_stage4_lora_a100.py
@@ -32,7 +32,7 @@
 # MAIA SFT VALIDATION
 # STAGE 4 - FULL ASSISTANT CONTROL
 #
-# GPT-2 MEDIUM RESPONSE-ONLY LoRA SFT
+# GPT-

In [17]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 9 - GPT-2 Medium Stage 4 LoRA
#
# CONTROL EXPERIMENT
#   Device: NVIDIA A100
#   Precision: FP32
#   Context: 1024
#   Micro batch: 1
#   Gradient accumulation: 8
#   Effective batch: 8
#   Epochs: 3
#
# The training script performs:
#   - frozen corpus verification
#   - A100/CUDA verification
#   - initial validation
#   - LoRA training
#   - periodic resumable checkpoints
#   - validation after every epoch
#   - best/final adapter preservation
# ============================================================

import os
import subprocess
import sys
from pathlib import Path


print("=" * 72)
print("MAIA LITE - SFT A100")
print("GPT-2 MEDIUM STAGE 4 LoRA CONTROL")
print("=" * 72)
print()

script = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora_a100.py"
)

if not script.is_file():
    raise RuntimeError(
        f"Training script not found: {script}"
    )

print("Project:")
print(f"  {SFT_PROJECT}")

print()
print("Training script:")
print(f"  {script}")

print()
print("Working directory:")
print(f"  {SFT_PROJECT}")

print()
print("Experiment:")
print("  Base model:             openai-community/gpt2-medium")
print("  Device:                 NVIDIA A100")
print("  Precision:              FP32")
print("  Context:                1024")
print("  Micro batch:            1")
print("  Gradient accumulation:  8")
print("  Effective batch:        8")
print("  Epochs:                 3")
print("  LoRA r/alpha/dropout:   16 / 32 / 0.05")

print()
print("=" * 72)
print("STARTING TRAINING")
print("=" * 72)
print()

env = os.environ.copy()

env["PYTHONUNBUFFERED"] = "1"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "-m",
        "src.train_stage4_lora_a100",
    ],
    cwd=str(SFT_PROJECT),
    env=env,
)

print()
print("=" * 72)

if result.returncode == 0:
    print("GPT-2 MEDIUM LoRA A100: COMPLETED")
else:
    print(
        "GPT-2 MEDIUM LoRA A100: FAILED "
        f"(exit code {result.returncode})"
    )

print("=" * 72)

if result.returncode != 0:
    raise RuntimeError(
        "A100 LoRA training process failed."
    )

MAIA LITE - SFT A100
GPT-2 MEDIUM STAGE 4 LoRA CONTROL

Project:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation

Training script:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/src/train_stage4_lora_a100.py

Working directory:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation

Experiment:
  Base model:             openai-community/gpt2-medium
  Device:                 NVIDIA A100
  Precision:              FP32
  Context:                1024
  Micro batch:            1
  Gradient accumulation:  8
  Effective batch:        8
  Epochs:                 3
  LoRA r/alpha/dropout:   16 / 32 / 0.05

STARTING TRAINING



KeyboardInterrupt: 

In [12]:
# ============================================================
# MAIA LITE - SFT A100
# Diagnostic - Import A100 Training Module
#
# Does NOT call main()
# Does NOT start training
# ============================================================

import os
import subprocess
import sys

print("=" * 72)
print("A100 LoRA - IMPORT DIAGNOSTIC")
print("=" * 72)
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

code = """
import sys
import traceback

print("Python:", sys.version)
print()

try:
    import src.train_stage4_lora_a100
    print("MODULE IMPORT: PASS")
except Exception:
    print("MODULE IMPORT: FAILED")
    print()
    traceback.print_exc()
    sys.exit(1)
"""

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        code,
    ],
    cwd=str(SFT_PROJECT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)

print("=" * 72)
print("Exit code:", result.returncode)
print("=" * 72)

A100 LoRA - IMPORT DIAGNOSTIC

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

MODULE IMPORT: PASS

Exit code: 0


In [13]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 9B - Run With Full Error Visibility
#
# Runs the SAME validated A100 training script.
# No source file is modified.
# ============================================================

import os
import subprocess
import sys


print("=" * 72)
print("MAIA LITE - SFT A100")
print("GPT-2 MEDIUM LoRA - FULL DIAGNOSTIC OUTPUT")
print("=" * 72)
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "-m",
        "src.train_stage4_lora_a100",
    ],
    cwd=str(SFT_PROJECT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)

finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()

print()
print("=" * 72)
print(f"PROCESS EXIT CODE: {return_code}")
print("=" * 72)

if return_code != 0:
    print()
    print(
        "Training process failed. "
        "The original traceback should now be visible above."
    )
else:
    print()
    print("Training completed successfully.")

MAIA LITE - SFT A100
GPT-2 MEDIUM LoRA - FULL DIAGNOSTIC OUTPUT


VERIFYING FROZEN STAGE 4 V4 CORPUS

train.jsonl
  expected: ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9
  actual:   ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9

validation.jsonl
  expected: 5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a
  actual:   5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a

test.jsonl
  expected: 03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834
  actual:   03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834

Frozen corpus verification: PASS

CUDA device: NVIDIA A100-SXM4-80GB
CUDA memory: 79.25 GiB
BF16 support: True

STAGE 4 LoRA TRAINING

Device: cuda
Model:  openai-community/gpt2-medium

Loading datasets...
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1317 > 1024). Running this sequence through the model will result in inde

In [14]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 10 - Remove Incompatible Optional torchao Package
#
# PEFT 0.21.0 detects torchao 0.10.0, but requires > 0.16.0.
# Stage 4 LoRA does not use torchao.
# ============================================================

import subprocess
import sys

print("=" * 72)
print("REMOVE INCOMPATIBLE OPTIONAL TORCHAO")
print("=" * 72)
print()

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=True,
)

print()
print("=" * 72)
print("TORCHAO REMOVAL: COMPLETE")
print("=" * 72)
print()
print("Restart the Colab runtime before continuing.")

REMOVE INCOMPATIBLE OPTIONAL TORCHAO


TORCHAO REMOVAL: COMPLETE

Restart the Colab runtime before continuing.


In [15]:
# ============================================================
# Check torchao state in a fresh Python subprocess
# ============================================================

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-c",
        """
import importlib.metadata

try:
    version = importlib.metadata.version("torchao")
    print("torchao:", version)
except importlib.metadata.PackageNotFoundError:
    print("torchao: NOT INSTALLED")

import torch
import transformers
import peft

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
"""
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)
print("Exit code:", result.returncode)

torchao: NOT INSTALLED
torch: 2.11.0+cu130
transformers: 5.17.0
peft: 0.21.0
CUDA: 13.0
GPU: NVIDIA A100-SXM4-80GB

Exit code: 0


In [16]:
# ============================================================
# MAIA LITE - SFT A100
# Post-Restart Recovery and Integrity Check
# ============================================================

from google.colab import drive
from pathlib import Path
import hashlib
import importlib.metadata
import torch


print("=" * 72)
print("MAIA LITE - SFT A100")
print("POST-RESTART RECOVERY")
print("=" * 72)
print()


# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

COLAB_ROOT = Path(
    "/content/drive/MyDrive/Colab_LLMs"
)

SFT_PROJECT = (
    COLAB_ROOT
    / "maia-sft-validation"
)

STAGE4 = (
    SFT_PROJECT
    / "data"
    / "stage4"
)

A100_SCRIPT = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora_a100.py"
)


# ------------------------------------------------------------
# SHA-256 helper
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

print()
print("Environment")
print("-" * 72)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available."
    )

gpu_name = torch.cuda.get_device_name(0)

print("PyTorch:     ", torch.__version__)
print("CUDA:        ", torch.version.cuda)
print("GPU:         ", gpu_name)
print(
    "GPU memory:  ",
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB",
)
print(
    "BF16:        ",
    torch.cuda.is_bf16_supported(),
)

if "A100" not in gpu_name.upper():
    raise RuntimeError(
        f"Expected A100, got: {gpu_name}"
    )


# ------------------------------------------------------------
# Python package state
# ------------------------------------------------------------

print()
print("Packages")
print("-" * 72)

for package in [
    "transformers",
    "peft",
    "accelerate",
]:
    version = importlib.metadata.version(package)

    print(
        f"{package:14s} {version}"
    )


try:
    torchao_version = (
        importlib.metadata.version("torchao")
    )

    raise RuntimeError(
        "torchao is still installed: "
        f"{torchao_version}"
    )

except importlib.metadata.PackageNotFoundError:
    print(
        f"{'torchao':14s} NOT INSTALLED — PASS"
    )


# ------------------------------------------------------------
# Frozen Stage 4 hashes
# ------------------------------------------------------------

print()
print("Frozen Stage 4")
print("-" * 72)

expected_hashes = {
    STAGE4 / "train.jsonl":
        "ee80a1efc74807d354761d9728cd498e9"
        "f543c84a5e568e4f3821d2fa48565f9",

    STAGE4 / "validation.jsonl":
        "5d8ea877e14105e06068f2d7de5ae55d"
        "265b4069af2c1b0762b162df1be1347a",

    STAGE4 / "test.jsonl":
        "03b7289a5841f79e1c95188f22615dca"
        "6789846aeddecc1b3c90f9713c918834",

    STAGE4 / "benchmark" / "stage4-v4-benchmark.jsonl":
        "9ba60ac68ad285fdb0e16eb2af876722d"
        "8745e8131203b4a090b51b6a5ad73c3",
}

for path, expected in expected_hashes.items():

    if not path.is_file():
        raise RuntimeError(
            f"Missing frozen artifact: {path}"
        )

    actual = sha256_file(path)

    status = (
        "PASS"
        if actual == expected
        else "FAIL"
    )

    print(
        f"{path.name:32s} {status}"
    )

    if actual != expected:
        raise RuntimeError(
            f"Hash mismatch: {path}"
        )


# ------------------------------------------------------------
# A100 training script
# ------------------------------------------------------------

print()
print("A100 training script")
print("-" * 72)

EXPECTED_A100_SHA256 = (
    "80edb17ae5c03c93cddc03db8bea3ad9"
    "62908d9921d65d7b95c5f4e0398d8a4d"
)

if not A100_SCRIPT.is_file():
    raise RuntimeError(
        f"Missing A100 script: {A100_SCRIPT}"
    )

actual_script_hash = sha256_file(
    A100_SCRIPT
)

print("Expected:")
print(EXPECTED_A100_SHA256)

print()
print("Actual:")
print(actual_script_hash)

if actual_script_hash != EXPECTED_A100_SHA256:
    raise RuntimeError(
        "A100 script hash mismatch."
    )

print()
print("A100 script integrity: PASS")


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print()
print("=" * 72)
print("POST-RESTART RECOVERY: PASS")
print("=" * 72)
print()
print("Environment recovered.")
print("Frozen Stage 4 corpus verified.")
print("A100 training script verified.")
print("Incompatible torchao absent.")
print()
print("Training was NOT started.")

MAIA LITE - SFT A100
POST-RESTART RECOVERY

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Environment
------------------------------------------------------------------------
PyTorch:      2.11.0+cu130
CUDA:         13.0
GPU:          NVIDIA A100-SXM4-80GB
GPU memory:   79.25 GiB
BF16:         True

Packages
------------------------------------------------------------------------
transformers   5.17.0
peft           0.21.0
accelerate     1.15.0
torchao        NOT INSTALLED — PASS

Frozen Stage 4
------------------------------------------------------------------------
train.jsonl                      PASS
validation.jsonl                 PASS
test.jsonl                       PASS
stage4-v4-benchmark.jsonl        PASS

A100 training script
------------------------------------------------------------------------
Expected:
80edb17ae5c03c93cddc03db8bea3ad962908d9921d65d7b95c5f4e0398d8a4d

Actual:
80edb17ae5c

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 9B - GPT-2 Medium Stage 4 LoRA
# Real-Time Output
#
# Runs the validated A100 training script.
# stdout and stderr are streamed live to the notebook.
# ============================================================

import os
import subprocess
import sys


print("=" * 72)
print("MAIA LITE - SFT A100")
print("GPT-2 MEDIUM LoRA - REAL-TIME TRAINING OUTPUT")
print("=" * 72)
print()

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# ------------------------------------------------------------
# Command
# ------------------------------------------------------------

command = [
    sys.executable,
    "-u",
    "-m",
    "src.train_stage4_lora_a100",
]

print("Project:")
print(f"  {SFT_PROJECT}")
print()

print("Command:")
print("  " + " ".join(command))
print()

print("Expected experiment:")
print("  Model:                  openai-community/gpt2-medium")
print("  Device:                 NVIDIA A100")
print("  Precision:              FP32")
print("  Context:                1024")
print("  Micro batch:            1")
print("  Gradient accumulation:  8")
print("  Effective batch:        8")
print("  Epochs:                 3")
print("  LoRA r/alpha/dropout:   16 / 32 / 0.05")

print()
print("=" * 72)
print("STARTING TRAINING")
print("=" * 72)
print()


# ------------------------------------------------------------
# Start process
# ------------------------------------------------------------

process = subprocess.Popen(
    command,
    cwd=str(SFT_PROJECT),
    env=env,

    # Merge stderr into stdout so traceback/warnings
    # are visible in the notebook.
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,

    # Text mode + line buffering.
    text=True,
    bufsize=1,
)


# ------------------------------------------------------------
# Stream output live
# ------------------------------------------------------------

try:
    if process.stdout is None:
        raise RuntimeError(
            "Could not capture training process output."
        )

    for line in iter(process.stdout.readline, ""):
        print(
            line,
            end="",
            flush=True,
        )

finally:
    if process.stdout is not None:
        process.stdout.close()


# ------------------------------------------------------------
# Wait for process
# ------------------------------------------------------------

return_code = process.wait()


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print()
print("=" * 72)
print(f"PROCESS EXIT CODE: {return_code}")
print("=" * 72)
print()

if return_code == 0:

    print(
        "GPT-2 MEDIUM LoRA A100: "
        "COMPLETED SUCCESSFULLY"
    )

else:

    print(
        "GPT-2 MEDIUM LoRA A100: FAILED"
    )

    raise RuntimeError(
        "Training process failed. "
        "See the complete traceback above."
    )

A saída de streaming foi truncada nas últimas 5000 linhas.
Epoch 3/3:  67%|██████▋   | 19968/30011 [1:07:16<1:01:26,  2.72it/s, avg=2.6383, loss=3.2454, lr=6.73e-07, update=10000/11256]
Checkpoint saved: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/a100-gpt2-medium-lora-checkpoints/checkpoint-010000

Epoch 3/3:  73%|███████▎  | 21968/30011 [1:13:59<27:39,  4.85it/s, avg=2.6362, loss=1.3351, lr=4.34e-07, update=10250/11256]
Checkpoint saved: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/a100-gpt2-medium-lora-checkpoints/checkpoint-010250

Epoch 3/3:  80%|███████▉  | 23968/30011 [1:20:43<20:48,  4.84it/s, avg=2.6354, loss=2.3982, lr=2.46e-07, update=10500/11256]
Checkpoint saved: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/a100-gpt2-medium-lora-checkpoints/checkpoint-010500

Epoch 3/3:  81%|████████  | 24340/30011 [1:22:00<19:17,  4.90it/s, avg=2.6356, loss=3.1934, lr=2.17e-07, update=10546/11256]
